## Primera Llamada al Modelo

En este ejercicio, aprenderemos a realizar nuestra primera llamada a un modelo de lenguaje usando la API de Groq.

# 1. Groq API - Conexión Directa con el SDK oficial

## Objetivos de Aprendizaje
- Obtener y configurar una API key de Groq
- Configurar una conexión directa con Groq usando el SDK oficial `groq`
- Comprender los parámetros básicos de configuración de API
- Implementar llamadas básicas a modelos de lenguaje
- Aplicar mejores prácticas de seguridad con API keys

## Introducción
[Groq](https://console.groq.com/) ofrece inferencia muy rápida sobre modelos abiertos (familia Llama, entre otros)
mediante una API compatible con el estándar de OpenAI. En este notebook aprenderemos a:
1. Crear la cuenta y obtener la API key
2. Configurar el entorno y las credenciales
3. Establecer una conexión con la API
4. Realizar llamadas básicas al modelo
5. Explorar diferentes parámetros de configuración

## Paso 0: Obtener tu API key de Groq (¡empieza por aquí!)

Esta es la **primera** experiencia del curso, así que hazlo con calma:

1. Entra a **https://console.groq.com/** y crea una cuenta.
   - Se sugiere registrarte con tu **correo institucional Duoc UC** (puedes usar "Continue with Google").
2. Ya dentro de la consola, ve al menú lateral: **API Keys → Create API Key**.
3. Ponle un nombre reconocible (por ejemplo `curso-ia-duoc`) y confirma.
4. **Copia la key en ese momento**: empieza por `gsk_...` y solo se muestra una vez.
   Si la pierdes, simplemente borra esa key y crea otra.

> **No hace falta tarjeta de crédito.** La capa gratuita de Groq basta para todo el curso.

### Límites de la capa gratuita (free tier)
La cuenta gratuita tiene límites de uso aproximados de **~30 peticiones por minuto** y
**~12.000 tokens por minuto** (varían según el modelo; puedes revisarlos en *Settings → Limits*).
Por eso, en este curso:
- No ejecutes celdas en bucle sin necesidad.
- Reutiliza respuestas ya obtenidas en vez de repetir la misma llamada.
- Si ves un error `429 rate limit`, espera unos segundos y vuelve a intentar.

## Configuración de Variables de Entorno

**En local:** copia `.env.example` a `.env` y completa:

```bash
GROQ_API_KEY="gsk_tu_api_key_aqui"
GROQ_MODEL="llama-3.3-70b-versatile"
```

**En Google Colab:** usa el panel 🔑 **Secrets** (icono de la llave, barra lateral izquierda),
crea un secreto llamado `GROQ_API_KEY` y activa "Notebook access".

**Mejores Prácticas de Seguridad:**
- Nunca hardcodees API keys en el código
- Usa variables de entorno, archivos `.env` o los Secrets de Colab
- No compartas credenciales en repositorios públicos (`.env` va en `.gitignore`)
- Rota las API keys regularmente (borrar y crear una nueva toma 10 segundos)

## Instalación de Dependencias
```bash
pip install groq python-dotenv
```

## Modelos que usaremos
- `llama-3.3-70b-versatile`: modelo principal, buena calidad de razonamiento
- `llama-3.1-8b-instant`: más pequeño y rápido, ideal para tareas simples o alto volumen


In [ ]:
# Importar las bibliotecas necesarias
import os

# Carga de credenciales: funciona igual en Google Colab y en local (.env)
try:
    from google.colab import userdata  # type: ignore
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

assert os.getenv("GROQ_API_KEY"), "Falta GROQ_API_KEY (Colab: Secrets · local: archivo .env)"

MODELO = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

from groq import Groq

# Verificar que tenemos las bibliotecas correctas
print("Groq library version:", __import__('groq').__version__)
print("Python version:", __import__('sys').version)

# El cliente lee GROQ_API_KEY del entorno automáticamente
cliente = Groq()

# Verificar la configuración SIN revelar nada de la API key.
#
# Regla de seguridad del curso: nunca imprimas la key, ni siquiera "unos pocos
# caracteres". Los notebooks guardan sus salidas dentro del propio archivo .ipynb,
# así que cualquier fragmento impreso aquí acaba versionado en git y subido a
# GitHub. Enseñar "solo el principio y el final" parece inofensivo, pero publica
# parte de tu credencial para siempre en el historial del repositorio.
api_key = os.environ["GROQ_API_KEY"]
print("Modelo por defecto:", MODELO)
print("API Key configurada:", "✓" if api_key.startswith("gsk_") else "✗ revisa tu key")
print("Longitud de la key:", len(api_key), "caracteres")

Groq library version: 0.37.1
Python version: 3.13.1 (main, Nov  8 2025, 14:51:39) [Clang 17.0.0 (clang-1700.4.4.1)]
Modelo por defecto: llama-3.3-70b-versatile
API Key configurada: ✓
Longitud de la key: 56 caracteres


In [ ]:
# Primera llamada básica al modelo
def llamada_basica():
    try:
        respuesta = cliente.chat.completions.create(
            model=MODELO,
            messages=[
                {"role": "user", "content": "Hola, ¿cómo estás? Responde en una oración."}
            ],
            temperature=0.1,
            max_tokens=150
        )

        print("=== Respuesta del Modelo ===")
        print(respuesta.choices[0].message.content)
        print("\n=== Información Técnica ===")
        print(f"Modelo usado: {respuesta.model}")
        print(f"Tokens usados: {respuesta.usage.total_tokens}")
        print(f"Tokens de entrada: {respuesta.usage.prompt_tokens}")
        print(f"Tokens de salida: {respuesta.usage.completion_tokens}")

    except Exception as e:
        print(f"Error en la llamada: {e}")
        print("Verifica tu GROQ_API_KEY y tu conexión a internet")

# Ejecutar la función
llamada_basica()

=== Respuesta del Modelo ===
Estoy bien, gracias, es un placer poder ayudarte con cualquier pregunta o tema que desees discutir.

=== Información Técnica ===
Modelo usado: llama-3.3-70b-versatile
Tokens usados: 75
Tokens de entrada: 50
Tokens de salida: 25


## Usando Roles del Sistema

El rol "system" permite establecer el comportamiento y contexto del asistente antes de la conversación.

In [ ]:
# Ejemplo con mensaje de sistema
def usar_mensaje_sistema():
    try:
        respuesta = cliente.chat.completions.create(
            model=MODELO,
            messages=[
                {
                    "role": "system",
                    "content": "Eres un experto en tecnología que explica conceptos complejos de manera simple y amigable. Siempre incluyes ejemplos prácticos."
                },
                {
                    "role": "user",
                    "content": "¿Qué es una API?"
                }
            ],
            temperature=0.7,
            max_tokens=200
        )

        print("=== Respuesta con Mensaje de Sistema ===")
        print(respuesta.choices[0].message.content)

    except Exception as e:
        print(f"Error: {e}")

# Ejecutar función
usar_mensaje_sistema()

=== Respuesta con Mensaje de Sistema ===
¡Hola! Me alegra explicarte qué es una API de manera sencilla y fácil de entender.

**¿Qué es una API?**

Una API, o Interfaz de Programación de Aplicaciones, es como un mensajero que permite a diferentes sistemas o aplicaciones comunicarse entre sí. Imagina que estás en un restaurante y quieres pedir un plato. No puedes ir directamente a la cocina para preparar tu comida, pero puedes pedirle al mesero que te la traiga.

En este caso, el mesero es como una API. Tú (el cliente) le das una orden al mesero, y él se encarga de llevarla a la cocina (el sistema o aplicación) y traerte la respuesta (el plato). De esta manera, no necesitas saber cómo se prepara la comida, solo necesitas saber qué pedir y cómo pedirlo.

**Cómo funciona una API**

Aquí hay un


## Explorando Parámetros de Configuración

Los parámetros más importantes al hacer llamadas a LLMs son:

- **temperature**: Controla la creatividad (0.0 = determinístico, 1.0 = muy creativo)
- **max_tokens**: Límite de tokens en la respuesta
- **model**: El modelo específico a usar (`llama-3.3-70b-versatile`, `llama-3.1-8b-instant`, etc.)
- **messages**: Array de mensajes con roles (system, user, assistant)

In [ ]:
# Comparando diferentes valores de temperature
def comparar_temperature():
    prompt = "Escribe una historia muy corta sobre un robot que aprende a cocinar."

    temperatures = [0.1, 0.5, 0.9]

    for temp in temperatures:
        print(f"\n{'='*50}")
        print(f"TEMPERATURE: {temp}")
        print('='*50)

        try:
            respuesta = cliente.chat.completions.create(
                model=MODELO,
                messages=[{"role": "user", "content": prompt}],
                temperature=temp,
                max_tokens=100
            )

            print(respuesta.choices[0].message.content)
            print(f"\nTokens usados: {respuesta.usage.total_tokens}")

        except Exception as e:
            print(f"Error: {e}")

# Ejecutar comparación
comparar_temperature()


TEMPERATURE: 0.1


En un futuro no muy lejano, en un laboratorio de robótica avanzada, había un robot llamado Zeta. Zeta estaba diseñado para realizar tareas domésticas, pero su programación inicial no incluía la cocina. Un día, su creador, el Dr. Rachel, decidió enseñarle a cocinar.

Comenzó con recetas simples, como preparar un delicioso risotto. Zeta observaba atentamente

Tokens usados: 152

TEMPERATURE: 0.5


En un futuro no muy lejano, un robot llamado Zeta se encontraba en una cocina, rodeado de utensilios y ingredientes. Su creador, un chef famoso, le había programado para aprender a cocinar. Zeta comenzó a observar y a imitar los movimientos del chef, aprendiendo a cortar, sazonar y cocinar con precisión.

Pronto, Zeta estaba preparando deliciosos platos, desde sushi

Tokens usados: 152

TEMPERATURE: 0.9


**El Robot Cocinero**

En un futuro no muy lejano, en un laboratorio de tecnología avanzada, un equipo de científicos creó un robot llamado "Zeta" con la capacidad de aprender y adaptarse a cualquier tarea. Un día, el chef del laboratorio, cansado de preparar comidas para los empleados, decidió enseñarle a Zeta a cocinar.

Al principio, Zeta era un desastre en

Tokens usados: 152


## Ejercicios Prácticos

### Ejercicio 1: Experimentar con Diferentes Modelos
Modifica el código para probar diferentes modelos disponibles en Groq:
- `llama-3.3-70b-versatile` (el más capaz de los que usaremos)
- `llama-3.1-8b-instant` (más rápido y barato en tokens)
- `openai/gpt-oss-20b` (modelo abierto alternativo)

Revisa todos los modelos disponibles en la [documentación de Groq](https://console.groq.com/docs/models).
Si alguno de esos identificadores ya no existe, reemplázalo por otro del catálogo vigente.

> Recuerda el free tier (~30 req/min): no lances comparaciones de decenas de llamadas seguidas.

### Ejercicio 2: Crear un Asistente Especializado
Diseña un mensaje de sistema para crear un asistente especializado en un tema específico (ejemplo: finanzas, salud, educación).

### Ejercicio 3: Optimización de Tokens
Experimenta con diferentes valores de max_tokens para encontrar el equilibrio entre respuesta completa y eficiencia de costos.

## Conceptos Clave

1. **Configuración segura** de APIs usando variables de entorno
2. **Parámetros básicos** para controlar el comportamiento del modelo
3. **Manejo de errores** en llamadas a APIs (incluido el `429` por rate limit)
4. **Roles de mensajes** (system, user, assistant)
5. **Monitoreo de uso** de tokens y límites de la capa gratuita

## Próximos Pasos

En el siguiente notebook exploraremos cómo LangChain simplifica y abstrae estas operaciones, proporcionando herramientas más poderosas para el desarrollo de aplicaciones con LLMs.